## StormEvents Cleaning v5.7 — Residual narrative fill (synchronous API)

v5.6 used the OpenAI Batch API to fill ~1 M rows; 1,014 requests failed with truncated JSON.
Root cause: the static schema asked the model to echo back a long existing narrative verbatim
inside the JSON, blowing past the 500-token limit.

Fix: build the response schema **per row** so it only requests the field(s) that are actually
missing. The existing narrative is passed as read-only context but never echoed in the output.

Input:  `./work/filled/StormEvents_narratives_filled.csv`  (output of v5.6)
Output: `./work/filled/StormEvents_narratives_filled_fin.csv`

### Import libraries

In [1]:
import pandas as pd
import numpy as np
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

### Connect to OpenAI

In [2]:
load_dotenv()
client = OpenAI()

### Load the v5.6 output and identify residual rows

In [3]:
IN_PATH  = "./work/filled/StormEvents_narratives_filled.csv"
OUT_PATH = "./work/filled/StormEvents_narratives_filled_fin.csv"

df = pd.read_csv(IN_PATH, low_memory=False)
print(f"Loaded: {df.shape}")

episode_empty = df["EPISODE_NARRATIVE"].isna()
event_empty   = df["EVENT_NARRATIVE"].isna()
either_empty  = episode_empty | event_empty

residual_idx = df.index[either_empty]
print(f"Residual rows (either narrative still empty): {len(residual_idx):,}")
print(f"  EPISODE_NARRATIVE empty: {episode_empty.sum():,}")
print(f"  EVENT_NARRATIVE   empty: {event_empty.sum():,}")

Loaded: (1830044, 36)
Residual rows (either narrative still empty): 1,014
  EPISODE_NARRATIVE empty: 2
  EVENT_NARRATIVE   empty: 1,012


### Configuration

Key fix vs. v5.7 first run: `_build_response_format()` and `_build_system_prompt()` are called
**per row** and only request the field(s) that are actually missing. The existing narrative is
passed as read-only context so the model never needs to echo it back — eliminating the
`Unterminated string` truncation failures on rows with long existing narratives.

In [4]:
RUN_NARRATIVE_API = True       # set False to dry-run without spending tokens
MODEL             = "gpt-4o-mini"
MAX_RETRIES       = 3          # per-row retry attempts on transient errors
RETRY_DELAY       = 5          # seconds between retries
RATE_LIMIT_SLEEP  = 0.5        # seconds between successful calls (~120 req/min)

CONTEXT_COLS = [
    "EVENT_TYPE", "EVENT_GROUP", "STATE", "CZ_NAME", "BEGIN_DATE_TIME", "END_DATE_TIME",
    "BEGIN_LOCATION", "END_LOCATION", "MAGNITUDE", "MAGNITUDE_TYPE", "TOR_F_SCALE",
    "TOR_LENGTH", "TOR_WIDTH", "FLOOD_CAUSE", "DAMAGE_PROPERTY", "DAMAGE_CROPS",
    "INJURIES_DIRECT", "INJURIES_INDIRECT", "DEATHS_DIRECT", "DEATHS_INDIRECT",
    "SOURCE", "WFO",
]


def _build_response_format(need_episode: bool, need_event: bool) -> dict:
    """Schema only requests the field(s) actually missing for this row."""
    props, required = {}, []
    if need_episode:
        props["episode_narrative"] = {"type": "string"}
        required.append("episode_narrative")
    if need_event:
        props["event_narrative"] = {"type": "string"}
        required.append("event_narrative")
    schema = {
        "type": "object",
        "properties": props,
        "required": required,
        "additionalProperties": False,
    }
    return {"type": "json_schema", "json_schema": {"name": "narrative", "schema": schema, "strict": True}}


def _build_system_prompt(need_episode: bool, need_event: bool) -> str:
    if need_episode and need_event:
        keys = (
            "two keys:\n"
            "  episode_narrative: 1 sentence on the broader weather episode/setup.\n"
            "  event_narrative:   1 sentence on this specific event and its impacts."
        )
    elif need_episode:
        keys = "one key:\n  episode_narrative: 1 sentence on the broader weather episode/setup."
    else:
        keys = "one key:\n  event_narrative: 1 sentence on this specific event and its impacts."
    return (
        "You write factual, concise NOAA Storm Events narratives from structured fields. "
        f"Return ONLY a JSON object with {keys}\n"
        "If an existing narrative is provided as context, use it to keep your output consistent "
        "but do NOT include it in the JSON response — only write the missing field(s). "
        "Use only the facts provided; do not invent places, names, or numbers not given. "
        "Plain past-tense prose, no preamble."
    )


def _row_to_prompt(row):
    facts = "; ".join(f"{c}={row[c]}" for c in CONTEXT_COLS if c in row and pd.notna(row[c]))
    parts = [f"Storm event fields: {facts}."]
    if pd.notna(row["EPISODE_NARRATIVE"]):
        parts.append(f"Context (existing episode_narrative — do NOT repeat in output): {row['EPISODE_NARRATIVE']}")
    if pd.notna(row["EVENT_NARRATIVE"]):
        parts.append(f"Context (existing event_narrative — do NOT repeat in output): {row['EVENT_NARRATIVE']}")
    return " ".join(parts)

### Fill residual rows via synchronous API

Each row gets up to `MAX_RETRIES` attempts. Per-row schema and system prompt are built based on
which field is missing, so the model never needs to echo back an existing (potentially long) narrative.

In [5]:
df_out      = df.copy()
filled_n    = 0
still_empty = []

if RUN_NARRATIVE_API:
    for i, idx in enumerate(residual_idx):
        row         = df_out.loc[idx]
        need_ep     = pd.isna(row["EPISODE_NARRATIVE"])
        need_ev     = pd.isna(row["EVENT_NARRATIVE"])
        prompt      = _row_to_prompt(row)
        sys_prompt  = _build_system_prompt(need_ep, need_ev)
        resp_format = _build_response_format(need_ep, need_ev)
        data        = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.chat.completions.create(
                    model=MODEL,
                    max_tokens=500,
                    response_format=resp_format,
                    messages=[
                        {"role": "system", "content": sys_prompt},
                        {"role": "user",   "content": prompt},
                    ],
                )
                data = json.loads(resp.choices[0].message.content)
                break
            except Exception as e:
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY * attempt)
                else:
                    print(f"  [FAIL] idx={idx} after {MAX_RETRIES} attempts: {e}")

        if data is not None:
            if need_ep:
                df_out.at[idx, "EPISODE_NARRATIVE"] = data.get("episode_narrative", "")
            if need_ev:
                df_out.at[idx, "EVENT_NARRATIVE"] = data.get("event_narrative", "")
            df_out.at[idx, "NARRATIVE_SOURCE"] = "llm_generated"
            filled_n += 1
        else:
            still_empty.append(idx)

        if (i + 1) % 100 == 0:
            print(f"  {i+1:,}/{len(residual_idx):,} processed — filled so far: {filled_n:,}")

        time.sleep(RATE_LIMIT_SLEEP)

    print(f"\nDone. Filled: {filled_n:,}  |  Still empty after retries: {len(still_empty):,}")
    if still_empty:
        print(f"Indices that could not be filled: {still_empty}")
else:
    print("RUN_NARRATIVE_API is False — skipping API calls.")

  100/1,014 processed — filled so far: 100
  200/1,014 processed — filled so far: 200
  300/1,014 processed — filled so far: 300
  400/1,014 processed — filled so far: 400
  500/1,014 processed — filled so far: 500
  600/1,014 processed — filled so far: 600
  700/1,014 processed — filled so far: 700
  800/1,014 processed — filled so far: 800
  900/1,014 processed — filled so far: 900
  1,000/1,014 processed — filled so far: 1,000

Done. Filled: 1,014  |  Still empty after retries: 0


### Sanity checks and save

In [6]:
assert len(df_out) == len(df),                                       "row count changed"
assert (df_out["EVENT_ID"].values == df["EVENT_ID"].values).all(),  "row order changed"

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUT_PATH, index=False)

print(f"Saved -> {OUT_PATH}  shape: {df_out.shape}")
print(f"Remaining empty EPISODE_NARRATIVE: {df_out['EPISODE_NARRATIVE'].isna().sum():,}")
print(f"Remaining empty EVENT_NARRATIVE:   {df_out['EVENT_NARRATIVE'].isna().sum():,}")

Saved -> ./work/filled/StormEvents_narratives_filled_fin.csv  shape: (1830044, 36)
Remaining empty EPISODE_NARRATIVE: 0
Remaining empty EVENT_NARRATIVE:   0
